# Method verification

Synthetic numerical checks for binary logistic LASSO, Elastic Net, and half-sample Stability Selection.

## Environment

In [1]:
from pathlib import Path
import json, os, sys, subprocess, shutil, platform
from importlib.metadata import version, PackageNotFoundError
from datetime import datetime, timezone

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    drive.mount('/content/drive')

candidates = []
if os.environ.get('CMPS355_ROOT'):
    candidates.append(Path(os.environ['CMPS355_ROOT']).expanduser())
if IN_COLAB:
    candidates.extend([Path('/content/drive/MyDrive/CMPS355_Research'), Path('/content/CMPS355_Research')])
candidates += [Path.cwd(), *Path.cwd().parents, Path.cwd() / 'CMPS355_Research']
ROOT = next((p.resolve() for p in candidates if (p / '.cmps355_root').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('CMPS355_Research not found. Extract the update in MyDrive/CMPS355_Research or set CMPS355_ROOT.')
sys.path.insert(0, str(ROOT / 'src')) if str(ROOT / 'src') not in sys.path else None
print('Project root:', ROOT)
print('Runtime:', 'Colab' if IN_COLAB else 'local', '| Python:', sys.version.split()[0])

Mounted at /content/drive
Project root: /content/drive/MyDrive/CMPS355_Research
Runtime: Colab | Python: 3.13.18


## Software versions

In [2]:
expected = {'numpy':'2.3.5','pandas':'2.2.3','scipy':'1.17.0','scikit-learn':'1.8.0'}
found = {}
for name in expected:
    try:
        found[name] = version(name)
    except PackageNotFoundError:
        found[name] = 'NOT_INSTALLED'
print('Installed packages:', found)
if any(found[k] != expected[k] for k in expected):
    print('Installing the exact reference-test versions from requirements.txt ...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(ROOT/'requirements.txt')], check=True)
    print('IMPORTANT: Restart the Colab runtime and run this notebook from Cell A again if packages were previously imported.')
else:
    print('All tested versions match.')

Installed packages: {'numpy': '2.4.6', 'pandas': '2.2.3', 'scipy': '1.16.3', 'scikit-learn': '1.6.1'}
Installing the exact reference-test versions from requirements.txt ...
IMPORTANT: Restart the Colab runtime and run this notebook from Cell A again if packages were previously imported.


## Logistic LASSO and Elastic Net

The implementation minimizes the binary logistic objective with unpenalized intercept:

$$\frac{1}{n}\sum_{i=1}^n[\log(1+e^{b+x_i^\top w})-y_i(b+x_i^\top w)]+\frac{\alpha\lVert w\rVert_1+(1-\alpha)\lVert w\rVert_2^2/2}{nC}.$$

The SAGA optimizer is compared with an independent constrained SciPy optimizer. References: [Tibshirani (1996)](https://doi.org/10.1111/j.2517-6161.1996.tb02080.x); [Zou and Hastie (2005)](https://doi.org/10.1111/j.1467-9868.2005.00503.x).

In [3]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from cmps355.method_verification import (
    SparseLogisticSpec, sparse_logistic_model, penalty_objective,
    independent_reference_solver, synthetic_binary_data, stability_selection
)

X, y, features = synthetic_binary_data(seed=19)
X_train, X_test = X[:120], X[120:]
y_train, y_test = y[:120], y[120:]

reference_rows = []
for method, alpha in [('LASSO logistic', 1.0), ('Elastic Net logistic', 0.5)]:
    spec = SparseLogisticSpec(C=0.16, l1_ratio=alpha, seed=42)
    pipe = sparse_logistic_model(spec)
    pipe.fit(X_train, y_train)
    scaled_train = pipe['scale'].transform(X_train)
    learned = pipe['logistic']
    saga_loss = penalty_objective(scaled_train, y_train, learned.coef_[0], learned.intercept_[0], spec)
    ref_weights, ref_intercept, ref_loss = independent_reference_solver(scaled_train, y_train, spec)
    loss_gap = saga_loss - ref_loss
    coef_gap = float(np.max(np.abs(learned.coef_[0] - ref_weights)))
    assert loss_gap > -1e-6 and loss_gap < 1e-4, (method, loss_gap)
    assert coef_gap < 0.02, (method, coef_gap)
    assert np.allclose(pipe['scale'].mean_, X_train.mean(axis=0)), 'Scaler leaked test rows'
    nonzero = [features[i] for i, value in enumerate(learned.coef_[0]) if abs(value) > 1e-5]
    reference_rows.append({
        'method': method, 'C': spec.C, 'alpha_L1': alpha,
        'SAGA_objective': float(saga_loss), 'independent_objective': float(ref_loss),
        'objective_gap': float(loss_gap), 'max_coef_gap': coef_gap,
        'selected_count': len(nonzero), 'selected_features': nonzero,
        'synthetic_test_AUROC': float(roc_auc_score(y_test, pipe.predict_proba(X_test)[:,1])),
        'reference_status': 'PASS'
    })
reference_table = pd.DataFrame(reference_rows)
display(reference_table.drop(columns='selected_features'))
for result in reference_rows:
    print(result['method'], '| nonzero synthetic features:', result['selected_features'])
print('Independent numerical optimization checks: PASS')

,method,C,alpha_L1,SAGA_objective,independent_objective,objective_gap,max_coef_gap,selected_count,synthetic_test_AUROC,reference_status
0,LASSO logistic,0.16,1.0,0.518295,0.518295,9.414691e-14,0.000001,3,0.884712,PASS
1,Elastic Net logistic,0.16,0.5,0.482329,0.482329,2.882694e-13,0.000001,8,0.902256,PASS


LASSO logistic | nonzero synthetic features: ['synthetic_00', 'synthetic_01', 'synthetic_02']
Elastic Net logistic | nonzero synthetic features: ['synthetic_00', 'synthetic_01', 'synthetic_02', 'synthetic_03', 'synthetic_04', 'synthetic_07', 'synthetic_09', 'synthetic_10']
Independent numerical optimization checks: PASS


## Stability Selection

Half-sample selection frequencies are computed for each fixed penalty, followed by a maximum across the penalty grid and a fixed selection threshold. The implementation does not include randomized Lasso weights. Reference: [Meinshausen and Bühlmann (2010)](https://doi.org/10.1111/j.1467-9868.2010.00740.x).

In [4]:
stability = stability_selection(X_train, y_train, features,
                               C_grid=[0.08, 0.13], n_subsamples=30,
                               threshold=0.8, seed=2026)
selection_table = pd.DataFrame({
    'feature': stability.feature_names,
    'P_selected_C_0.08': stability.per_C_frequencies[0],
    'P_selected_C_0.13': stability.per_C_frequencies[1],
    'max_selection_probability': stability.max_frequencies,
    'stable': [f in stability.stable_features for f in stability.feature_names],
}).sort_values('max_selection_probability', ascending=False)
display(selection_table)
print('Stable synthetic features:', stability.stable_features)
print('Subsample size:', stability.half_sample_size)
assert stability.half_sample_size == len(X_train)//2
assert all(np.asarray(stability.max_frequencies) >= np.asarray(stability.per_C_frequencies[0]))
assert all(np.asarray(stability.max_frequencies) >= np.asarray(stability.per_C_frequencies[1]))
print('Half-sample stability selection invariants: PASS')

,feature,P_selected_C_0.08,P_selected_C_0.13,max_selection_probability,stable
0,synthetic_00,1.000000,1.000000,1.000000,True
1,synthetic_01,0.166667,0.833333,0.833333,True
2,synthetic_02,0.000000,0.433333,0.433333,False
4,synthetic_04,0.000000,0.066667,0.066667,False
11,synthetic_11,0.000000,0.033333,0.033333,False
9,synthetic_09,0.000000,0.033333,0.033333,False
3,synthetic_03,0.000000,0.000000,0.000000,False
5,synthetic_05,0.000000,0.000000,0.000000,False
7,synthetic_07,0.000000,0.000000,0.000000,False
6,synthetic_06,0.000000,0.000000,0.000000,False


Stable synthetic features: ['synthetic_00', 'synthetic_01']
Subsample size: 60
Half-sample stability selection invariants: PASS


## Official R package availability

StableMate and TENTACLES are checked for installation and recorded source revisions. Package presence is distinct from method reproduction.

In [5]:
r_script = shutil.which('Rscript')
r_status = {'Rscript': r_script, 'preflight': 'NOT_RUN', 'stdout': '', 'stderr': ''}
if r_script:
    p = subprocess.run([r_script, str(ROOT/'r'/'check_official_packages.R')],
                       capture_output=True, text=True, timeout=60)
    r_status.update(preflight='DIAGNOSTIC_ONLY' if p.returncode==0 else 'ERROR',
                    stdout=p.stdout, stderr=p.stderr[:1200])
    print(p.stdout)
    if p.stderr:
        print('R diagnostic stderr:', p.stderr[:500])
else:
    r_status['preflight'] = 'PENDING_R_INSTALL'
    print('Rscript is absent here. Python numerical checks passed; official StableMate/TENTACLES tests are PENDING.')

remote_heads = {}
for package, url in {
    'StableMate': 'https://github.com/meiosis97/StableMate.git',
    'TENTACLES': 'https://github.com/Giomu/TENTACLES.git'
}.items():
    if shutil.which('git') is None:
        remote_heads[package] = {'status': 'NOT_AVAILABLE', 'sha': None}
        continue
    try:
        result = subprocess.run(['git', 'ls-remote', url, 'HEAD'],
                                capture_output=True, text=True, timeout=15)
        sha = result.stdout.split()[0] if result.returncode==0 and result.stdout.strip() else None
        remote_heads[package] = {'status': 'RESOLVED_FOR_THIS_RUN' if sha else 'UNAVAILABLE', 'sha': sha}
    except (OSError, subprocess.TimeoutExpired):
        remote_heads[package] = {'status': 'UNAVAILABLE', 'sha': None}
print('Remote revisions (NOT installed):', remote_heads)

R_VERSION=4.6.1
R_PLATFORM=x86_64-pc-linux-gnu
StableMate_STATUS=NOT_INSTALLED
TENTACLES_STATUS=NOT_INSTALLED
TENTACLES_R_VERSION_OK=YES
R_METHOD_SMOKE_TESTED=NO

Remote revisions (NOT installed): {'StableMate': {'status': 'RESOLVED_FOR_THIS_RUN', 'sha': 'cf0fe9f344c1756b6f89fd33f9fdae1a7f35b24a'}, 'TENTACLES': {'status': 'RESOLVED_FOR_THIS_RUN', 'sha': '069a26271e3885e015df25baa40a7a1140827d5f'}}


## Provenance

In [6]:
from cmps355.project_checks import sha256_file

report = {
    'stage': '02_published_method_verification',
    'generated_utc': datetime.now(timezone.utc).isoformat(),
    'project': 'CMPS355_Research',
    'data_used': 'SYNTHETIC_ONLY',
    'real_patient_data_modelled': False,
    'platform': platform.platform(), 'python': sys.version.split()[0],
    'dependencies': {pkg: version(pkg) for pkg in expected},
    'method_module_sha256': sha256_file(ROOT/'src/cmps355/method_verification.py'),
    'method_tests': reference_rows,
    'stability_selection': stability.as_dict(),
    'r_preflight': r_status, 'official_remote_heads': remote_heads,
    'status': {
        'lasso_logistic_numeric_reference': 'PASS_SYNTHETIC',
        'elastic_net_logistic_numeric_reference': 'PASS_SYNTHETIC',
        'basic_half_sample_stability_selection': 'PASS_SYNTHETIC',
        'stablemate_author_implementation': 'PENDING_OFFICIAL_R_EXECUTION',
        'tentacles_author_implementation': 'PENDING_OFFICIAL_R_EXECUTION',
        'real_cohort_modeling': 'NOT_STARTED',
    },
}
out = ROOT/'runs'/'method_verification'/f"verification_{datetime.now(timezone.utc):%Y%m%dT%H%M%S%fZ}.json"
out.parent.mkdir(parents=True, exist_ok=True)
with out.open('x', encoding='utf-8') as handle:
    json.dump(report, handle, ensure_ascii=False, indent=2)
print('Report saved:', out.relative_to(ROOT))
print('STAGE 2 PYTHON REFERENCE CHECKS: PASS')
print('OFFICIAL R PACKAGE METHOD VERIFICATION: PENDING')
print('Next: pinned R package installation + official examples, then leakage-safe LOCO pilot.')

Report saved: runs/method_verification/verification_20261106T215928187928Z.json
STAGE 2 PYTHON REFERENCE CHECKS: PASS
OFFICIAL R PACKAGE METHOD VERIFICATION: PENDING
Next: pinned R package installation + official examples, then leakage-safe LOCO pilot.
